# Validation
Re-derives the hash values directly from each source table and compares them against what is stored in `hashed_values`. Two checks:

1. **Row count check** — source table row count vs. output table row count (per table).
2. **Hash re-computation check** — recompute `primary_key_hash_value`, `composite_keys_hash_value` and `hash_value` straight from the source table and diff them against the stored values, row by row.

A worked, hand-written SQL example for a single table is included at the end.

## Imports

In [0]:
%run ./00_config

## Helper functions
Identical standardization/hashing logic to `02_hash_generation`, repeated here so this notebook can run independently as a pure read-only check — it never writes to `hashed_values`.

In [0]:
def hash_expr(algorithm, expr):
    """Return a hash column expression for the given algorithm."""
    algorithm = algorithm.strip().lower()
    if algorithm == "md5":
        return F.md5(expr)
    elif algorithm in ("sha256", "sha-256"):
        return F.sha2(expr, 256)
    else:
        raise ValueError(f"Unsupported hash_algorithm: {algorithm}")


def get_hashable_columns(catalog, schema, table):
    """
    (column_name, data_type) for every column on the table, per information_schema.columns,
    ORDERED BY ordinal_position (i.e. COLUMN_ID / the table's declared column order),
    excluding any column whose name contains '___c' (metadata columns).
    """
    cols_df = spark.sql(f"""
        SELECT column_name, data_type
        FROM {catalog}.information_schema.columns
        WHERE table_schema = '{schema}'
          AND table_name   = '{table}'
          AND column_name NOT LIKE '%___c%'
        ORDER BY ordinal_position
    """)
    return [(r["column_name"], r["data_type"]) for r in cols_df.collect()]


NULL_PLACEHOLDER = "NULL"


def standardize_column(colname, data_type):

    dtype = data_type.upper()
    col = F.col(colname)

    if dtype.startswith("DECIMAL") or dtype in ("DOUBLE", "FLOAT"):
        standardized = F.format_number(col.cast("decimal(38,10)"), 10)
    elif dtype == "DATE":
        standardized = F.date_format(col, "yyyy-MM-dd")
    elif dtype.startswith("TIMESTAMP"):
        standardized = F.date_format(col, "yyyy-MM-dd HH:mm:ss.SSSSSS")
    elif dtype == "BOOLEAN":
        standardized = F.lower(col.cast("string"))
    elif dtype.startswith("STRING") or dtype.startswith("VARCHAR") or dtype.startswith("CHAR"):
        standardized = F.trim(col.cast("string"))
    else:
        standardized = col.cast("string")

    return F.coalesce(standardized, F.lit(NULL_PLACEHOLDER))


def concat_columns(col_specs):
    return F.concat_ws("||", *[standardize_column(name, dtype) for name, dtype in col_specs])

## Load active config

In [0]:
output_table = f"{tgt_catalog}.{tgt_schema}.{tgt_table}"

config_rows = (
    spark.table(config_table)
    .filter(F.col("is_active") == 1)
    .collect()
)

## Check 1 — row counts (source vs. output)

In [0]:
count_results = []

for row in config_rows:
    src_catalog = row["catalog"]
    src_schema  = row["schema"]
    src_table   = row["table"]
    full_table_name = f"{src_catalog}.{src_schema}.{src_table}"

    src_count = spark.table(full_table_name).count()
    out_count = (
        spark.table(output_table)
        .filter(
            (F.col("catalog") == src_catalog)
            & (F.col("schema") == src_schema)
            & (F.col("table") == src_table)
        )
        .count()
    )

    count_results.append((src_catalog, src_schema, src_table, src_count, out_count, src_count == out_count))

count_df = spark.createDataFrame(
    count_results,
    schema="catalog STRING, schema STRING, table STRING, "
           "source_row_count LONG, output_row_count LONG, counts_match BOOLEAN"
)

display(count_df)

## Check 2 — recompute hashes from source and diff against the output table
For each table: rebuild `expected_primary_key_hash_value`, `expected_composite_keys_hash_value` and `expected_hash_value` directly from the source, join to the matching output rows on `primary_key`, and flag any row where a stored hash doesn't match the freshly-computed one (or where the row is missing entirely from the output table).

In [0]:
mismatch_dfs = []
summary_results = []

for row in config_rows:
    src_catalog    = row["catalog"]
    src_schema     = row["schema"]
    src_table      = row["table"]
    primary_key    = row["primary_key"].strip()
    composite_keys = [c.strip() for c in row["composite_keys"].split(",")]
    hash_algorithm = row["hash_algorithm"].strip().lower()
    full_table_name = f"{src_catalog}.{src_schema}.{src_table}"

    all_columns = get_hashable_columns(src_catalog, src_schema, src_table)  # [(name, data_type), ...] in COLUMN_ID order
    if not all_columns:
        print(f"WARNING: no hashable columns found for {full_table_name}, skipping")
        continue

    col_types = dict(all_columns)
    src_df = spark.table(full_table_name)

    pk_value_expr = concat_columns([(primary_key, col_types.get(primary_key, "STRING"))])
    ck_value_expr = concat_columns([(c, col_types.get(c, "STRING")) for c in composite_keys])
    row_value_expr = concat_columns(all_columns)

    expected_df = src_df.select(
        pk_value_expr.alias("primary_key"),
        hash_expr(hash_algorithm, pk_value_expr).alias("expected_primary_key_hash_value"),
        hash_expr(hash_algorithm, ck_value_expr).alias("expected_composite_keys_hash_value"),
        hash_expr(hash_algorithm, row_value_expr).alias("expected_hash_value"),
    )

    actual_df = (
        spark.table(output_table)
        .filter(
            (F.col("catalog") == src_catalog)
            & (F.col("schema") == src_schema)
            & (F.col("table") == src_table)
        )
        .select("primary_key", "primary_key_hash_value", "composite_keys_hash_value", "hash_value")
    )

    compared_df = expected_df.join(actual_df, on="primary_key", how="left")

    mismatches_df = compared_df.filter(
        F.col("primary_key_hash_value").isNull()
        | (F.col("primary_key_hash_value") != F.col("expected_primary_key_hash_value"))
        | (F.col("composite_keys_hash_value") != F.col("expected_composite_keys_hash_value"))
        | (F.col("hash_value") != F.col("expected_hash_value"))
    ).select(
        F.lit(src_catalog).alias("catalog"),
        F.lit(src_schema).alias("schema"),
        F.lit(src_table).alias("table"),
        "primary_key",
        "primary_key_hash_value", "expected_primary_key_hash_value",
        "composite_keys_hash_value", "expected_composite_keys_hash_value",
        "hash_value", "expected_hash_value",
    )

    mismatch_count = mismatches_df.count()
    summary_results.append((src_catalog, src_schema, src_table, mismatch_count, mismatch_count == 0))
    if mismatch_count > 0:
        mismatch_dfs.append(mismatches_df)

summary_df = spark.createDataFrame(
    summary_results,
    schema="catalog STRING, schema STRING, table STRING, mismatch_count LONG, all_hashes_match BOOLEAN"
)

display(summary_df)

### Row-level mismatch detail
Only populated if `mismatch_count > 0` for one or more tables above.

In [0]:
if mismatch_dfs:
    all_mismatches_df = mismatch_dfs[0]
    for df in mismatch_dfs[1:]:
        all_mismatches_df = all_mismatches_df.unionByName(df)
    display(all_mismatches_df.limit(200))
else:
    print("No mismatches found across any active table.")

In [0]:
# spark.sql("""
# WITH source_recomputed AS (
#     SELECT
#         CAST(CASE_SURR_ID AS STRING) AS primary_key,
#         md5(CAST(CASE_SURR_ID AS STRING)) AS expected_primary_key_hash_value,
#         md5(concat_ws('||',
#             CAST(CASE_SURR_ID AS STRING), CAST(PATIENT_SURR_ID AS STRING), CAST(CASE_TYPE AS STRING),
#             CAST(HEALTHCARE_FACILITY AS STRING), CAST(PATIENT_AC_NO AS STRING), CAST(DIAG_CODE AS STRING),
#             CAST(DIAG_TYPE AS STRING), CAST(MDC AS STRING), CAST(DRG AS STRING)
#         )) AS expected_composite_keys_hash_value,
#         -- NOTE: replace this column list with every non-"___c" column on the table
#         md5(concat_ws('||',
#             CAST(CASE_SURR_ID AS STRING), CAST(PATIENT_SURR_ID AS STRING), CAST(CASE_TYPE AS STRING),
#             CAST(HEALTHCARE_FACILITY AS STRING), CAST(PATIENT_AC_NO AS STRING), CAST(DIAG_CODE AS STRING),
#             CAST(DIAG_TYPE AS STRING), CAST(MDC AS STRING), CAST(DRG AS STRING)
#         )) AS expected_hash_value
#     FROM iics.casemix.healthcare_ingestion_test
# )
# SELECT
#     s.primary_key,
#     o.primary_key_hash_value,        s.expected_primary_key_hash_value,
#     o.composite_keys_hash_value,     s.expected_composite_keys_hash_value,
#     o.hash_value,                    s.expected_hash_value,
#     CASE
#         WHEN o.primary_key IS NULL THEN 'MISSING_IN_OUTPUT'
#         WHEN  o.primary_key_hash_value      = s.expected_primary_key_hash_value
#           AND o.composite_keys_hash_value   = s.expected_composite_keys_hash_value
#           AND o.hash_value                  = s.expected_hash_value
#         THEN 'MATCH'
#         ELSE 'MISMATCH'
#     END AS validation_status
# FROM source_recomputed s
# LEFT JOIN iics.casemix.hashed_values o
#     ON  o.primary_key = s.primary_key
#     AND o.catalog = 'iics' AND o.schema = 'casemix' AND o.table = 'healthcare_ingestion_test'
# WHERE o.primary_key IS NULL
#    OR o.primary_key_hash_value    != s.expected_primary_key_hash_value
#    OR o.composite_keys_hash_value != s.expected_composite_keys_hash_value
#    OR o.hash_value                != s.expected_hash_value
# """).display()